In [29]:
"""
generate_cps230_synthetic_data.py
----------------------------------
Sinh dữ liệu synthetic cho Star Schema "CPS 230 Operational Resilience"
(Fact_Incidents + Dim_Date, Dim_Operation, Dim_Tolerance_Limit, Dim_Vendor)
phục vụ portfolio Data Analyst.

Điểm mấu chốt: Fact_Incidents được sinh theo 10 "kịch bản nghiệp vụ / lỗi dữ
liệu" khác nhau (xem CASE_WEIGHTS bên dưới), để khi bạn viết SQL/DAX profiling
trong portfolio, bạn có đủ case thật để show ra insight và QC logic.

Output (trong ./output/):
    Dim_Date.csv
    Dim_Operation.csv
    Dim_Tolerance_Limit.csv
    Dim_Vendor.csv
    Fact_Incidents.csv                 (raw, CHƯA làm sạch — có đủ lỗi)
    Duplicate_Incidents_Check.csv      (kết quả của 1 QC query tìm incident bị log trùng)
"""

import os
import random
import numpy as np
import pandas as pd

# --------------------------------------------------------------------------
# 0. CONFIG
# --------------------------------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

N_BASE_INCIDENTS = 1500          # số incident "gốc" trước khi nhân bản duplicate
START_DATE, END_DATE = "2024-01-01", "2025-12-31"

# Ghi file CSV ngay tại thư mục chứa file .py này (không tạo folder con).
# __file__ không tồn tại khi chạy trong Jupyter/Colab/REPL -> fallback về cwd
try:
    OUTPUT_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    OUTPUT_DIR = os.getcwd()

# --------------------------------------------------------------------------
# 1. DIM_DATE
# --------------------------------------------------------------------------
def build_dim_date(start=START_DATE, end=END_DATE) -> pd.DataFrame:
    dates = pd.date_range(start, end, freq="D")
    df = pd.DataFrame({"Full_Date": dates})
    df["Date_Key"] = df["Full_Date"].dt.strftime("%Y%m%d").astype(int)
    df["Day_Of_Week"] = df["Full_Date"].dt.day_name()
    df["Month"] = df["Full_Date"].dt.month
    df["Quarter"] = df["Full_Date"].dt.quarter
    df["Year"] = df["Full_Date"].dt.year
    return df[["Date_Key", "Full_Date", "Day_Of_Week", "Month", "Quarter", "Year"]]


# --------------------------------------------------------------------------
# 2. DIM_OPERATION
# --------------------------------------------------------------------------
def build_dim_operation() -> pd.DataFrame:
    ops = [
        (1, "Mobile Banking App - Login/Auth",       "Digital Channels", "Tier 1"),
        (2, "Real-Time Payments (NPP)",               "Payments",         "Tier 1"),
        (3, "Internet Banking - Funds Transfer",      "Digital Channels", "Tier 1"),
        (4, "Card Payments Gateway",                  "Payments",         "Tier 1"),
        (5, "BPAY / Bill Payment",                    "Payments",         "Tier 1"),
        (6, "ATM Network Switch",                     "Payments",         "Tier 1"),
        (7, "Credit Card Application Online",         "Retail Banking",   "Tier 2"),
        (8, "Home Loan Origination Portal",           "Retail Banking",   "Tier 2"),
        (9, "Transaction History / Statements",       "Digital Channels", "Tier 2"),
        (10, "Customer Onboarding (eKYC)",            "Retail Banking",   "Tier 2"),
    ]
    return pd.DataFrame(
        ops, columns=["Operation_Key", "Operation_Name", "Business_Unit", "Criticality_Level"]
    )


# --------------------------------------------------------------------------
# 3. DIM_TOLERANCE_LIMIT  (ngưỡng CPS 230 - chỉ khối lượng ý nghĩa nhất với Tier 1)
# --------------------------------------------------------------------------
def build_dim_tolerance_limit(dim_operation: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for i, op in dim_operation.iterrows():
        if op["Criticality_Level"] == "Tier 1":
            max_dt = random.choice([15, 20, 30])         # phút - APRA-style limit
            max_txn = random.choice([500, 1000, 2000])
        else:
            max_dt = random.choice([60, 90, 120])
            max_txn = random.choice([2000, 5000, 8000])
        rows.append([i + 1, max_dt, max_txn, op["Operation_Key"]])
    return pd.DataFrame(
        rows,
        columns=["Limit_Key", "Max_Allowed_Downtime_Mins", "Max_Allowed_Failed_Txns", "Operation_Key"],
    )


# --------------------------------------------------------------------------
# 4. DIM_VENDOR
# --------------------------------------------------------------------------
def build_dim_vendor() -> pd.DataFrame:
    vendors = [
        (1, "Internal / In-house",   "N/A",      "Active"),
        (2, "CloudCore Hosting",     "Platinum", "Active"),
        (3, "PayGate Solutions",     "Gold",     "Active"),
        (4, "SecureAuth ID",         "Gold",     "Active"),
        (5, "NetLink Telco",         "Silver",   "Active"),
        (6, "OTP Verify Co",         "Silver",   "Under Review"),
        (7, "DataStream Analytics",  "Gold",     "Active"),
        (8, "LegacySys Partners",    "Bronze",   "Expired"),
    ]
    return pd.DataFrame(vendors, columns=["Vendor_Key", "Vendor_Name", "SLA_Tier", "Contract_Status"])


# --------------------------------------------------------------------------
# 5. FACT_INCIDENTS — logic sinh theo từng "kịch bản"
# --------------------------------------------------------------------------
# Giải thích từng case (đối chiếu Downtime_Minutes vs MTTR_Minutes):
#
#   successful          : downtime > 0, MTTR >= downtime            -> xử lý thành công, bình thường
#   self_resolved        : downtime > 0, MTTR = 0                    -> issue tự hết (auto-failover / self-heal),
#                           không cần con người can thiệp
#   logging_error        : downtime > 0, 0 < MTTR < downtime         -> lỗi ghi nhận: ticket bị đóng sớm hơn
#                           thời điểm hệ thống thực sự phục hồi
#   negative_downtime    : downtime < 0                              -> lỗi dữ liệu (timestamp start/end bị đảo)
#   ghost_impact          : downtime = 0 nhưng Impacted_Txns/Financial_Loss > 0
#                           -> lỗi dữ liệu HOẶC suy giảm hiệu năng (degradation) không được tính là downtime
#   false_alert           : downtime = 0 nhưng MTTR > 0              -> ticket được mở nhưng không có outage
#                           thực sự (false positive alert / monitoring noise)
#   outlier_downtime      : downtime cực lớn bất thường (vd nhầm đơn vị giây thay vì phút)
#                           -> lỗi nhập liệu, cần audit
#   extended_mttr_outlier : MTTR cực lớn so với downtime (vd ticket bị bỏ quên nhiều ngày)
#                           -> lỗi quy trình vận hành, cần audit
#   missing_data          : thiếu Vendor_Key và/hoặc Financial_Loss_AUD -> lỗi pipeline/ETL upstream
#   duplicate_source       : incident bị ghi nhận trùng 2 lần (cùng Incident_ID, khác Incident_SK)
#                           -> lỗi hệ thống logging (double event) - dùng để demo QC/dedup

CASE_WEIGHTS = {
    "successful":            0.62,
    "self_resolved":         0.12,
    "logging_error":         0.06,
    "negative_downtime":     0.02,
    "ghost_impact":          0.03,
    "false_alert":           0.03,
    "outlier_downtime":      0.02,
    "extended_mttr_outlier": 0.03,
    "missing_data":          0.04,
    "duplicate_source":      0.03,
}
CASE_NAMES = list(CASE_WEIGHTS.keys())
CASE_PROBS = list(CASE_WEIGHTS.values())

VENDOR_CLASSIFICATION_MAP = {
    1: "Internal Infrastructure Failure",   # Internal / In-house
}
DEFAULT_VENDOR_CLASSIFICATIONS = [
    "Third-Party/Vendor Outage",
    "Network/Telco Failure",
    "Software Deployment Bug",
]
GENERIC_CLASSIFICATIONS = [
    "Internal Infrastructure Failure",
    "Software Deployment Bug",
    "Cyber/Security Incident",
    "Planned Maintenance Overrun",
]


def _txn_rate_for_operation(op_row) -> float:
    """Giao dịch/phút trung bình khi outage - Tier 1 traffic cao hơn Tier 2."""
    base = 60 if op_row["Criticality_Level"] == "Tier 1" else 15
    return base


def _avg_txn_value(op_row) -> float:
    """Giá trị tổn thất quy đổi mỗi giao dịch ảnh hưởng (AUD), theo nhóm dịch vụ."""
    mapping = {
        "Payments": 35,
        "Digital Channels": 12,
        "Retail Banking": 20,
    }
    return mapping.get(op_row["Business_Unit"], 15)


def _pick_case() -> str:
    return np.random.choice(CASE_NAMES, p=CASE_PROBS)


def _pick_classification(vendor_key: int) -> str:
    if vendor_key == 1:
        return "Internal Infrastructure Failure"
    return random.choice(DEFAULT_VENDOR_CLASSIFICATIONS + ["Third-Party/Vendor Outage"])


def build_fact_incidents(dim_date, dim_operation, dim_vendor, dim_tolerance, n=N_BASE_INCIDENTS):
    date_keys = dim_date["Date_Key"].tolist()
    # Tier 1 operations chiếm tỉ trọng incident cao hơn (traffic lớn hơn -> nhiều sự cố hơn)
    op_weights = dim_operation["Criticality_Level"].map({"Tier 1": 0.75, "Tier 2": 0.25})
    op_weights = (op_weights / op_weights.sum()).tolist()
    tol_by_op = dim_tolerance.set_index("Operation_Key")

    rows = []
    incident_sk = 1

    for i in range(1, n + 1):
        incident_id = f"INC-{i:06d}"
        date_key = random.choice(date_keys)
        op_row = dim_operation.sample(1, weights=op_weights).iloc[0]
        vendor_row = dim_vendor.sample(1).iloc[0]
        is_peak = np.random.rand() < 0.35
        case = _pick_case()

        txn_rate = _txn_rate_for_operation(op_row) * (1.8 if is_peak else 1.0)
        avg_txn_value = _avg_txn_value(op_row)

        vendor_key = vendor_row["Vendor_Key"]
        financial_loss = None
        is_invalid = False

        # ---- Base downtime (phút), lấy từ phân phối lệch phải ----
        base_downtime = round(np.random.exponential(scale=22) + 3, 1)

        if case == "successful":
            downtime = base_downtime
            mttr = round(downtime + np.random.exponential(scale=8), 1)
            # ~10% các case Tier1 "thành công" vẫn có thể breach ngưỡng CPS230
            if op_row["Criticality_Level"] == "Tier 1" and np.random.rand() < 0.10:
                limit = tol_by_op.loc[op_row["Operation_Key"], "Max_Allowed_Downtime_Mins"]
                downtime = round(limit + np.random.uniform(5, 25), 1)
                mttr = round(downtime + np.random.exponential(scale=8), 1)

        elif case == "self_resolved":
            downtime = base_downtime
            mttr = 0.0

        elif case == "logging_error":
            downtime = base_downtime
            mttr = round(downtime * np.random.uniform(0.3, 0.9), 1)  # ticket closed too early
            is_invalid = True

        elif case == "negative_downtime":
            downtime = -round(base_downtime, 1)                       # start/end timestamp bị đảo
            mttr = round(base_downtime + np.random.exponential(scale=8), 1)
            is_invalid = True

        elif case == "ghost_impact":
            downtime = 0.0
            mttr = 0.0
            is_invalid = True

        elif case == "false_alert":
            downtime = 0.0
            mttr = round(np.random.uniform(2, 15), 1)                 # alert noise, engineer vẫn tạo/đóng ticket
            is_invalid = True

        elif case == "outlier_downtime":
            downtime = round(base_downtime * np.random.uniform(50, 150), 1)  # nhầm đơn vị giây<->phút
            mttr = round(base_downtime + np.random.exponential(scale=8), 1)
            is_invalid = True

        elif case == "extended_mttr_outlier":
            downtime = base_downtime
            mttr = round(downtime * np.random.uniform(15, 60), 1)     # ticket bị bỏ quên nhiều ngày
            is_invalid = True

        elif case == "missing_data":
            downtime = base_downtime
            mttr = round(downtime + np.random.exponential(scale=8), 1)
            vendor_key = None if np.random.rand() < 0.5 else vendor_key
            is_invalid = True

        elif case == "duplicate_source":
            downtime = base_downtime
            mttr = round(downtime + np.random.exponential(scale=8), 1)
            is_invalid = True

        # ---- Impacted transactions & financial loss ----
        eff_downtime = max(downtime, 0)
        impacted_txns = int(max(eff_downtime, 0) * txn_rate * np.random.uniform(0.7, 1.3))
        if case == "ghost_impact":
            impacted_txns = int(np.random.uniform(50, 400))  # có ảnh hưởng dù downtime = 0

        if case != "missing_data" or np.random.rand() > 0.5:
            financial_loss = round(
                impacted_txns * avg_txn_value * np.random.uniform(0.8, 1.2)
                + eff_downtime * np.random.uniform(15, 40),  # chi phí nhân sự xử lý
                2,
            )
        # missing_data: 50% khả năng Financial_Loss_AUD = None (đã set None ở trên mặc định)

        classification = _pick_classification(vendor_key if vendor_key is not None else 1)
        if case == "negative_downtime" or case == "outlier_downtime":
            classification = "Data/Logging Anomaly"

        # Limit_Key = khóa nối trực tiếp Fact_Incidents <-> Dim_Tolerance_Limit (đúng theo ERD gốc)
        limit_key = tol_by_op.loc[op_row["Operation_Key"], "Limit_Key"]

        row = {
            "Incident_SK": incident_sk,
            "Incident_ID": incident_id,
            "Date_Key": date_key,
            "Operation_Key": op_row["Operation_Key"],
            "Limit_Key": limit_key,
            "Vendor_Key": vendor_key,
            "Downtime_Minutes": downtime,
            "MTTR_Minutes": mttr,
            "Impacted_Txns_Count": impacted_txns,
            "Financial_Loss_AUD": financial_loss,
            "Is_Peak_Hour": bool(is_peak),
            "Incident_Classification": classification,
            "Is_Invalid_Column": is_invalid,
            "Data_Quality_Case": case,   # cột phụ trợ - phục vụ demo QC, có thể drop trước khi publish
        }
        rows.append(row)
        incident_sk += 1

        # ---- Nhân bản record cho case duplicate_source ----
        if case == "duplicate_source":
            dup_row = row.copy()
            dup_row["Incident_SK"] = incident_sk
            # incident bị log trùng: SLIGHT lag về thời gian ghi nhận -> MTTR chênh nhẹ
            dup_row["MTTR_Minutes"] = round(row["MTTR_Minutes"] + np.random.uniform(-2, 2), 1)
            rows.append(dup_row)
            incident_sk += 1

    return pd.DataFrame(rows)


# --------------------------------------------------------------------------
# 6. DUPLICATE_INCIDENTS_CHECK — mô phỏng kết quả của 1 QC query
#    (GROUP BY Incident_ID HAVING COUNT(*) > 1)
# --------------------------------------------------------------------------
def build_duplicate_check(fact_incidents: pd.DataFrame) -> pd.DataFrame:
    dup_ids = fact_incidents["Incident_ID"][
        fact_incidents["Incident_ID"].duplicated(keep=False)
    ].unique()
    dup_df = fact_incidents[fact_incidents["Incident_ID"].isin(dup_ids)].copy()
    cols = [
        "Incident_ID",
        "Date_Key",
        "Downtime_Minutes",
        "Financial_Loss_AUD",
        "Impacted_Txns_Count",
        "Is_Peak_Hour",
    ]
    return dup_df[cols].sort_values("Incident_ID").reset_index(drop=True)


# --------------------------------------------------------------------------
# 7. MAIN
# --------------------------------------------------------------------------
def main():
    dim_date = build_dim_date()
    dim_operation = build_dim_operation()
    dim_tolerance = build_dim_tolerance_limit(dim_operation)
    dim_vendor = build_dim_vendor()
    fact_incidents = build_fact_incidents(dim_date, dim_operation, dim_vendor, dim_tolerance)
    duplicate_check = build_duplicate_check(fact_incidents)

    dim_date.to_csv(os.path.join(OUTPUT_DIR, "Dim_Date.csv"), index=False)
    dim_operation.to_csv(os.path.join(OUTPUT_DIR, "Dim_Operation.csv"), index=False)
    dim_tolerance.to_csv(os.path.join(OUTPUT_DIR, "Dim_Tolerance_Limit.csv"), index=False)
    dim_vendor.to_csv(os.path.join(OUTPUT_DIR, "Dim_Vendor.csv"), index=False)
    fact_incidents.to_csv(os.path.join(OUTPUT_DIR, "Fact_Incidents.csv"), index=False)
    duplicate_check.to_csv(os.path.join(OUTPUT_DIR, "Duplicate_Incidents_Check.csv"), index=False)

    print(f"Fact_Incidents rows: {len(fact_incidents)}")
    print("Case distribution:")
    print(fact_incidents["Data_Quality_Case"].value_counts())
    print(f"\nDuplicate incidents found: {duplicate_check['Incident_ID'].nunique()} IDs "
          f"({len(duplicate_check)} rows)")
    print(f"\nFiles written to: {OUTPUT_DIR}")


if __name__ == "__main__":
    main()

Fact_Incidents rows: 1542
Case distribution:
Data_Quality_Case
successful               907
self_resolved            189
logging_error             99
duplicate_source          84
missing_data              71
extended_mttr_outlier     47
ghost_impact              47
false_alert               43
negative_downtime         29
outlier_downtime          26
Name: count, dtype: int64

Duplicate incidents found: 42 IDs (84 rows)

Files written to: C:\Users\ThinkPad\Documents\NAB_Appilication


In [30]:
fact_incidents_df = pd.read_csv("Fact_Incidents.csv")
fact_incidents_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1542 entries, 0 to 1541
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Incident_SK              1542 non-null   int64  
 1   Incident_ID              1542 non-null   object 
 2   Date_Key                 1542 non-null   int64  
 3   Operation_Key            1542 non-null   int64  
 4   Limit_Key                1542 non-null   int64  
 5   Vendor_Key               1510 non-null   float64
 6   Downtime_Minutes         1542 non-null   float64
 7   MTTR_Minutes             1542 non-null   float64
 8   Impacted_Txns_Count      1542 non-null   int64  
 9   Financial_Loss_AUD       1502 non-null   float64
 10  Is_Peak_Hour             1542 non-null   bool   
 11  Incident_Classification  1542 non-null   object 
 12  Is_Invalid_Column        1542 non-null   bool   
 13  Data_Quality_Case        1542 non-null   object 
dtypes: bool(2), float64(4), 

In [28]:
Duplicate_Incidents_Check_df = pd.read_csv("Duplicate_Incidents_Check.csv")
Duplicate_Incidents_Check_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 84 entries, 0 to 83
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Incident_ID          84 non-null     object 
 1   Date_Key             84 non-null     int64  
 2   Downtime_Minutes     84 non-null     float64
 3   Financial_Loss_AUD   84 non-null     float64
 4   Impacted_Txns_Count  84 non-null     int64  
 5   Is_Peak_Hour         84 non-null     bool   
dtypes: bool(1), float64(2), int64(2), object(1)
memory usage: 3.5+ KB
